<a href="https://colab.research.google.com/github/aycaaozturk/AML-project/blob/main/AML_sample_simplification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Select one representative AML sample per patient
#
# Priority:
#   1. Bone marrow sample          -> -09
#   2. Blood-derived sample        -> -03
#   3. Additional blood sample     -> -04
#   4. Other suffixes              -> review manually
#
# Input:
#   sample_train_imputed.csv
#
# Outputs:
#   sample_train_one_sample_per_patient.csv
#   sample_train_patients_for_manual_review.csv
#   sample_train_selection_log.csv
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# 1. File paths
# ------------------------------------------------------------

# Change this path to the location of your CSV file in Google Drive.
INPUT_PATH = Path(
    '/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/'
    'sample_split_and_imputed/clean/sample_train_imputed_clean.csv'
)

OUTPUT_DIR = Path(   '/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/'
    'simplified/sample_train'
)

SELECTED_OUTPUT_PATH = (
    OUTPUT_DIR / "sample_train_one_sample_per_patient.csv"
)

REVIEW_OUTPUT_PATH = (
    OUTPUT_DIR / "sample_train_patients_for_manual_review.csv"
)

LOG_OUTPUT_PATH = (
    OUTPUT_DIR / "sample_train_selection_log.csv"
)


In [ ]:
# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

PATIENT_ID_COL = "PATIENT_ID"
SAMPLE_ID_COL = "SAMPLE_ID"

# Lower number = higher priority
SUFFIX_PRIORITY = {
    "-09": 1,   # Bone marrow-derived sample
    "-03": 2,   # Blood-derived sample
    "-04": 3,   # Additional blood-derived sample
}

ELIGIBLE_SUFFIXES = set(SUFFIX_PRIORITY.keys())

# Choose how patients with no -09, -03, or -04 sample should be handled.
#
# "review":
#     Do not include them automatically in the simplified dataset.
#     Save them in a separate review file.
#
# "keep_first":
#     Keep one of the other samples, but mark it as requiring review.
#
# "exclude":
#     Exclude them without keeping one of their samples.
OTHER_SAMPLE_POLICY = "review"


# ------------------------------------------------------------
# 3. Load dataset
# ------------------------------------------------------------

df = pd.read_csv(INPUT_PATH)

print(f"Original number of rows: {len(df)}")
print(f"Unique patients: {df[PATIENT_ID_COL].nunique()}")
print(f"Unique samples: {df[SAMPLE_ID_COL].nunique()}")

Original number of rows: 711
Unique patients: 620
Unique samples: 711


In [ ]:
# ------------------------------------------------------------
# 4. Validate required columns
# ------------------------------------------------------------

required_columns = {
    PATIENT_ID_COL,
    SAMPLE_ID_COL,
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        "The following required columns are missing: "
        f"{sorted(missing_columns)}"
    )


# Check for missing patient or sample IDs
missing_patient_ids = df[PATIENT_ID_COL].isna().sum()
missing_sample_ids = df[SAMPLE_ID_COL].isna().sum()

if missing_patient_ids > 0 or missing_sample_ids > 0:
    raise ValueError(
        f"Missing PATIENT_ID values: {missing_patient_ids}\n"
        f"Missing SAMPLE_ID values: {missing_sample_ids}\n"
        "Resolve these missing identifiers before sample selection."
    )


# Convert IDs to strings and remove accidental spaces
df[PATIENT_ID_COL] = df[PATIENT_ID_COL].astype(str).str.strip()
df[SAMPLE_ID_COL] = df[SAMPLE_ID_COL].astype(str).str.strip()


In [ ]:
# ------------------------------------------------------------
# 5. Extract sample suffix
# ------------------------------------------------------------

# Example:
# TARGET-20-PABLDZ-09 -> -09
df["SAMPLE_SUFFIX"] = df[SAMPLE_ID_COL].str.extract(
    r"(-\d+)$",
    expand=False
)

rows_without_suffix = df["SAMPLE_SUFFIX"].isna()

if rows_without_suffix.any():
    print(
        "\nWarning: Some SAMPLE_ID values do not end with a numeric suffix."
    )
    print(
        df.loc[
            rows_without_suffix,
            [PATIENT_ID_COL, SAMPLE_ID_COL]
        ].head(20)
    )


# Assign priority
df["SAMPLE_PRIORITY"] = (
    df["SAMPLE_SUFFIX"]
    .map(SUFFIX_PRIORITY)
    .fillna(np.inf)
)

df["IS_ELIGIBLE_SAMPLE"] = (
    df["SAMPLE_SUFFIX"].isin(ELIGIBLE_SUFFIXES)
)



In [ ]:
# ------------------------------------------------------------
# 7. Select one representative sample per patient
# ------------------------------------------------------------

selected_rows = []
review_rows = []
selection_log = []

for patient_id, patient_samples in df.groupby(
    PATIENT_ID_COL,
    sort=False
):

    patient_samples = patient_samples.copy()

    eligible_samples = patient_samples[
        patient_samples["IS_ELIGIBLE_SAMPLE"]
    ].copy()

    available_suffixes = sorted(
        patient_samples["SAMPLE_SUFFIX"]
        .dropna()
        .unique()
        .tolist()
    )

    available_sample_ids = patient_samples[
        SAMPLE_ID_COL
    ].tolist()

    # --------------------------------------------------------
    # Decision tree:
    # Diagnostic/eligible samples available
    # --------------------------------------------------------
    if not eligible_samples.empty:

        # Sort by the specified priority:
        # -09 first, then -03, then -04
        eligible_samples = eligible_samples.sort_values(
            by=[
                "SAMPLE_PRIORITY",
                SAMPLE_ID_COL
            ],
            ascending=True
        )

        chosen_row = eligible_samples.iloc[0].copy()
        selected_rows.append(chosen_row)

        chosen_suffix = chosen_row["SAMPLE_SUFFIX"]

        if chosen_suffix == "-09":
            reason = (
                "Selected bone marrow-derived sample (-09), "
                "the highest-priority eligible sample."
            )

        elif chosen_suffix == "-03":
            reason = (
                "No -09 sample was available; selected "
                "blood-derived sample (-03)."
            )

        elif chosen_suffix == "-04":
            reason = (
                "No -09 or -03 sample was available; selected "
                "additional blood-derived sample (-04)."
            )

        else:
            reason = "Selected eligible sample."

        selection_log.append({
            PATIENT_ID_COL: patient_id,
            "N_AVAILABLE_SAMPLES": len(patient_samples),
            "AVAILABLE_SAMPLE_IDS": "; ".join(available_sample_ids),
            "AVAILABLE_SUFFIXES": "; ".join(available_suffixes),
            "SELECTED_SAMPLE_ID": chosen_row[SAMPLE_ID_COL],
            "SELECTED_SUFFIX": chosen_suffix,
            "SELECTION_STATUS": "Automatically selected",
            "SELECTION_REASON": reason
        })

    # --------------------------------------------------------
    # Decision tree:
    # No eligible -09, -03, or -04 sample available
    # --------------------------------------------------------
    else:

        patient_samples["REVIEW_REASON"] = (
            "No eligible -09, -03, or -04 sample was available."
        )

        review_rows.append(patient_samples)

        if OTHER_SAMPLE_POLICY == "keep_first":

            chosen_row = patient_samples.sort_values(
                SAMPLE_ID_COL
            ).iloc[0].copy()

            selected_rows.append(chosen_row)

            selection_log.append({
                PATIENT_ID_COL: patient_id,
                "N_AVAILABLE_SAMPLES": len(patient_samples),
                "AVAILABLE_SAMPLE_IDS": "; ".join(available_sample_ids),
                "AVAILABLE_SUFFIXES": "; ".join(available_suffixes),
                "SELECTED_SAMPLE_ID": chosen_row[SAMPLE_ID_COL],
                "SELECTED_SUFFIX": chosen_row["SAMPLE_SUFFIX"],
                "SELECTION_STATUS": "Kept but requires review",
                "SELECTION_REASON": (
                    "No -09, -03, or -04 sample was available. "
                    "The first alternative sample was retained."
                )
            })

        else:

            selection_log.append({
                PATIENT_ID_COL: patient_id,
                "N_AVAILABLE_SAMPLES": len(patient_samples),
                "AVAILABLE_SAMPLE_IDS": "; ".join(available_sample_ids),
                "AVAILABLE_SUFFIXES": "; ".join(available_suffixes),
                "SELECTED_SAMPLE_ID": pd.NA,
                "SELECTED_SUFFIX": pd.NA,
                "SELECTION_STATUS": "Manual review required",
                "SELECTION_REASON": (
                    "No eligible -09, -03, or -04 sample was available."
                )
            })



In [ ]:
# ------------------------------------------------------------
# 8. Create output DataFrames
# ------------------------------------------------------------

selected_df = pd.DataFrame(selected_rows).reset_index(drop=True)

if review_rows:
    review_df = pd.concat(
        review_rows,
        ignore_index=True
    )
else:
    review_df = pd.DataFrame(columns=list(df.columns) + ["REVIEW_REASON"])

selection_log_df = pd.DataFrame(selection_log)


In [ ]:
# ------------------------------------------------------------
# 9. Final validation
# ------------------------------------------------------------

# Every retained patient must have exactly one row
duplicated_selected_patients = selected_df[
    PATIENT_ID_COL
].duplicated(keep=False)

if duplicated_selected_patients.any():
    duplicate_ids = selected_df.loc[
        duplicated_selected_patients,
        PATIENT_ID_COL
    ].unique()

    raise RuntimeError(
        "Selection failed: some retained patients still have "
        f"multiple rows: {duplicate_ids}"
    )


assert (
    len(selected_df)
    == selected_df[PATIENT_ID_COL].nunique()
), "The selected dataset does not contain exactly one row per patient."

In [ ]:
# ------------------------------------------------------------
# 10. Remove temporary helper columns if desired
# ------------------------------------------------------------

helper_columns = [
    "SAMPLE_PRIORITY",
    "IS_ELIGIBLE_SAMPLE",
    "IS_DIAGNOSTIC"
]

selected_df = selected_df.drop(
    columns=helper_columns,
    errors="ignore"
)

review_df = review_df.drop(
    columns=helper_columns,
    errors="ignore"
)


# ------------------------------------------------------------
# 11. Save outputs
# ------------------------------------------------------------

selected_df.to_csv(
    SELECTED_OUTPUT_PATH,
    index=False
)

review_df.to_csv(
    REVIEW_OUTPUT_PATH,
    index=False
)

selection_log_df.to_csv(
    LOG_OUTPUT_PATH,
    index=False
)


# ------------------------------------------------------------
# 12. Print summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SAMPLE SELECTION SUMMARY")
print("=" * 60)

print(f"Original sample rows: {len(df)}")
print(f"Original unique patients: {df[PATIENT_ID_COL].nunique()}")
print(f"Selected patient rows: {len(selected_df)}")
print(
    "Patients requiring manual review: "
    f"{review_df[PATIENT_ID_COL].nunique() if not review_df.empty else 0}"
)

print("\nSelected suffix counts:")
print(
    selected_df["SAMPLE_SUFFIX"]
    .value_counts(dropna=False)
)

print("\nSelection status counts:")
print(
    selection_log_df["SELECTION_STATUS"]
    .value_counts(dropna=False)
)

print("\nFiles saved:")
print(f"Selected dataset: {SELECTED_OUTPUT_PATH}")
print(f"Manual review dataset: {REVIEW_OUTPUT_PATH}")
print(f"Selection log: {LOG_OUTPUT_PATH}")


SAMPLE SELECTION SUMMARY
Original sample rows: 711
Original unique patients: 620
Selected patient rows: 620
Patients requiring manual review: 0

Selected suffix counts:
SAMPLE_SUFFIX
-09    505
-03    114
-04      1
Name: count, dtype: int64

Selection status counts:
SELECTION_STATUS
Automatically selected    620
Name: count, dtype: int64

Files saved:
Selected dataset: /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/simplified/sample_train/sample_train_one_sample_per_patient.csv
Manual review dataset: /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/simplified/sample_train/sample_train_patients_for_manual_review.csv
Selection log: /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/simplified/sample_train/sample_train_selection_log.csv
